# HW 2: Discrete Heterogeneity

Part 1: Re-estimate the multinomial logit brand choice model with the last brand purchased variable on the Yogurt data, but using a 2 type discrete heterogeneity distribution.  For now put the heterogeneity only on the intercepts. The last assignment had you form an individual specific likelihood function that took an NT by 1 vector and collapsed it to N by 1. For this assignment, first form a likelihood function that is NT by M. Then take the products across T within the first dimension to collapse it into an N by M matrix with the likelihood for each individual if they were each type. Then, take the weighted average of the types over the 2nd dimension to form the individual specific likelihood function which is N by 1.

Part 2: Now try allowing all of the parameters to be type specific (still use 2 types).  Also, try allowing for 4 types but restricting types to vary only by intercepts.

Part 3: Numerically solve for cross price elasticities in the homogenous multinomial logit model and the two type finite support heterogeneity model.  To do so, simulate a small change in the price of brand 1.  Also try it by simulating a small price change for brand 2. Compare elasticities between the homogenous and heterogenous models.

In [1]:
# Imports
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
print(df.head())

   hh  qty        exp  nopurch  b1  b2  b3  b4    p1    p2    p3    p4  f1  \
0   1    2  40.900002        0   0   0   0   1  0.11  0.08  0.06  0.08   0   
1   1    0   8.830000        1   0   0   0   0  0.11  0.09  0.05  0.08   0   
2   1    0   3.880000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
3   1    0   0.780000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
4   1    0  39.240002        1   0   0   0   0  0.13  0.09  0.05  0.08   0   

   f2  f3  f4  tripnum  
0   0   0   0        1  
1   0   0   0        2  
2   0   0   0        3  
3   0   0   0        4  
4   0   0   0        5  


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)

# Create prev_purch variable
df_long["brand_num"] = df_long[["brand_1", "brand_2", "brand_3", "brand_4"]].idxmax(axis=1).str.replace("brand_", "").astype(int)
trip_choice = (
    df_long.loc[df_long["b"] == 1, ["hh", "tripnum", "brand_num"]]
    .rename(columns={"brand_num": "chosen_brand"})
)
trip_hist = (
    df_long[["hh", "tripnum"]]
    .drop_duplicates()
    .sort_values(["hh", "tripnum"])
    .merge(trip_choice, on=["hh", "tripnum"], how="left")
)
trip_hist["prev_purch"] = (
    trip_hist.groupby("hh")["chosen_brand"]
    .transform(lambda x: x.ffill().shift(1))
    .fillna(0)
    .astype(int)
)
df_long = df_long.merge(
    trip_hist[["hh", "tripnum", "prev_purch"]],
    on=["hh", "tripnum"],
    how="left",
)

df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4,brand_num,prev_purch
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0,1,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0,2,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0,3,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1,4,0
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0,1,4


### I. 2-type discrete heterogeneity on intercepts

Suppose there are $J$ products and $C$ latent classes. We let brand intercepts $\beta_{0cj}$ ($j=1,\ldots,J$) be class-specific and $\tilde{k}$ other covariates $\beta_1, \ldots, \beta_{\tilde{k}}$ be common across classes. Then, class $c$ has the covariate vector $\vec\beta_c = [\beta_{0c1}, \ldots, \beta_{0cJ}, \beta_1, \cdots, \beta_{\tilde{k}}]^\intercal \in \mathbb{R}^{k}$ where $k=J+\tilde{k}$ is the number of parameters in each covariate vector.

In the following implementation, I create a matrix $B \in \mathbb{R}^{k \times C}$ that stores the covariate vector of class $c$ as its $c$th column (note that $B$ stores all the covariate coefficients to be estimated):
$$
B = \begin{bmatrix} \vec\beta_1 & \cdots & \vec\beta_C \end{bmatrix} = 
\begin{bmatrix} 
\beta_{011} & \cdots & \beta_{0C1} \\
\vdots & \ddots & \vdots \\
\beta_{01J} & \cdots & \beta_{0CJ} \\
\beta_1 & \cdots & \beta_1 \\ 
\vdots & \ddots & \vdots \\
\beta_{\tilde{k}} & \cdots & \beta_{\tilde{k}} \\ 
\end{bmatrix}
$$
Recall that $X \in \mathbb{R}^{NTJ \times k}$ is the design matrix ($NT$ is the number of household-trips made, or equivalently, the number of choice occasions observed). Then $V = XB \in \mathbb{R}^{NTJ \times C}$ stores the utility $v_{itj\mid c}$ of household $i$ in trip $t$ choosing product $j$ if they were of class $c$:
$$
V = \begin{bmatrix}
v_{111\mid 1} & \cdots & v_{111\mid C} \\
\vdots & \ddots & \vdots \\
v_{11J\mid 1} & \cdots & v_{11J\mid C} \\
v_{121\mid 1} & \cdots & v_{121\mid C} \\
\vdots & \ddots & \vdots \\
v_{itj\mid 1} & \cdots & v_{itj\mid C} \\
\vdots & \ddots & \vdots \\
v_{NTJ\mid 1} & \cdots & v_{NTJ\mid C} \\
\end{bmatrix}
$$
We desire an $N \times C$ matrix that contains the likelihoods of each household choices as if they were of each class. To do so, we first reshape $V$ to shape $NT \times J \times C$ and collapse across the second dimension using $1+\sum_{j=1}^J \exp(v_{itj})$. Call the resulting matrix $L_\text{den} \in \mathbb{R}^{NT \times C}$—this is the denominator of the MNL formula. Letting $Y \in \mathbb{R}^{NTJ}$ be the binary vector of whether product $j$ was chosen for each row $itj$, form $\tilde{Y} \in \mathbb{R}^{NTJ \times C}$ that simply copies $Y$ in its $C$ columns. Then, $V \odot \tilde{Y}$ (element-wise matrix multiplication) and reshaping to shape $NT \times J \times C$ before collapsing across the second dimension with a regular sum returns a matrix containing the utility obtained by household $i$ in trip $t$ due to their observed choice, if they were of class $c$. Exponentiating each value gives us $L_\text{num} \in \mathbb{R}^{NT \times C}$—the numerator of the MNL formula. Element-wise division of $L_\text{num}$ over $L_\text{den}$ creates $L^{(3)} \in \mathbb{R}^{NT \times C}$, where $L^{(3)}_{itc}$ is the probability of observing household $i$ choose the product they chose in trip $t$, if they were of class $c$. Taking the product over all the trips $t$ made by household $i$ gives $L^{(2)} \in \mathbb{R}^{N \times C}$: $L^{(2)}_{hc} = \prod_t L^{(3)}_{itc}$.

Now let $\mu=[\mu_1, \ldots, \mu_C]$ be the proportions of each class ($\mu_c \geq 0, \sum_c \mu_c = 1$). To avoid the non-negativity and normalization constraints, we estimate $\{\tilde\mu_1, \ldots, \tilde\mu_C\}$ where $\tilde\mu_c \in \mathbb{R}$ for $c = 2, \ldots, C$ and $\tilde\mu_1=0$. Then, we recover $\mu_c = \frac{\exp(\tilde\mu_c)}{\sum_c \exp(\tilde\mu_c)}$. To obtain the likelihoods of each household, we compute $L^{(1)} = L^{(2)}\mu \in \mathbb{R}^{N}$, which takes a $\mu$-weighted sum over classes.

Finally, we use maximum likelihood estimation on $\sum_{i=1}^N \log L^{(1)}_i$ to obtain parameter estimates for $B$ and $\mu$.

In [4]:
def _unpack_theta(theta, k, n_classes, n_heterogenous):
    """
    Unpack theta into B (k x C) and free mu parameters (length C-1).

    Theta ordering: [heterogeneous coefs, common coefs, mu_free].
    Heterogeneous coefficients are column-stacked by class.
    """
    theta = np.asarray(theta).reshape(-1)
    n_hetero = int(n_heterogenous)
    n_common = k - n_hetero
    hetero_size = n_hetero * n_classes
    beta_size = hetero_size + n_common

    B = np.zeros((k, n_classes), dtype=float)

    if n_hetero > 0:
        B_hetero = theta[:hetero_size].reshape(n_hetero, n_classes, order="F")
        B[:n_hetero, :] = B_hetero

    if n_common > 0:
        B_common = theta[hetero_size:beta_size].reshape(n_common)
        B[n_hetero:, :] = B_common[:, None]

    mu_free = theta[beta_size:].reshape(-1)
    return B, mu_free


def _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous):
    """Negative log-likelihood for finite-mixture MNL with outside option,
    computed at the household level (vector of length n_households).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts  # NT
    B, mu_free = _unpack_theta(theta, k, n_classes, n_heterogenous=n_heterogenous)

    # L1: likelihood per hh-trip choice occasion, for each class
    V = (X @ B).reshape(n_choices, n_alts, n_classes, order="C")            # Utility tensor: (NT, J, C)
    V0 = np.concatenate([np.zeros((n_choices, 1, n_classes)), V], axis=1)   # add outside option with utility 0
    Y1 = Y.reshape(n_choices, n_alts, 1, order="C")
    chosen_v = np.multiply(Y1, V).sum(axis=1)   # (NT, C): Chosen utility per (choice occasion, class)
    log_L3 = chosen_v - logsumexp(V0, axis=1)

    # L2: likelihood per household, for each class
    hh_trip = hh_ids.reshape(n_choices, n_alts, order="C")[:, 0]
    _, hh_index = np.unique(hh_trip, return_inverse=True)
    log_L2 = np.column_stack([
        np.bincount(hh_index, weights=log_L3[:, c])
        for c in range(n_classes)
    ])

    # L1: likelihood per household, integrating over classes w.r.t. mu
    mu_raw = np.concatenate(([0.0], mu_free)) if n_classes > 1 else np.array([0.0])
    prop_class = mu_raw - logsumexp(mu_raw)     # back out class proportions
    log_L1 = logsumexp(log_L2 + prop_class[None, :], axis=1)

    return -log_L1


def _negloglik(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous):
    """Summed negative log-likelihood"""
    log_L1 = _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
    return np.sum(log_L1)


def score_matrix(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous, eps=1e-6):
    """
    Numerical score matrix of household-level negative log-likelihood contributions.
    Household-level aggregation (so this is cluster-robust to household correlation).
    Shape: (n_households, n_params).
    """
    theta = np.asarray(theta).reshape(-1)
    p = theta.size

    base = _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
    n_households = base.size
    S = np.empty((n_households, p), dtype=float)

    # Numerical differentiation with central difference approximation
    for k in range(p):
        step = eps * max(1.0, abs(theta[k]))
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[k] += step
        theta_minus[k] -= step

        f_plus = _negloglik_hh(theta_plus, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
        f_minus = _negloglik_hh(theta_minus, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
        S[:, k] = (f_plus - f_minus) / (2.0 * step)

    return S


def print_mle_output(output, covariate_vars, heterogenous_vars, robust_se, individual_var):
    """Prints MLE results in aligned tables, separated by class."""
    B_hat = np.asarray(output['opt_beta'])
    n_classes = B_hat.shape[1]
    k = B_hat.shape[0]

    se_theta = np.asarray(output['se_theta'])
    ci_theta = np.asarray(output['ci_theta'])

    if heterogenous_vars is None:
        n_hetero = k
    else:
        n_hetero = len(heterogenous_vars)

    n_common = k - n_hetero
    hetero_size = n_hetero * n_classes

    se_beta = np.zeros((k, n_classes), dtype=float)
    ci_beta = np.zeros((k, n_classes, 2), dtype=float)

    if n_hetero > 0:
        se_beta[:n_hetero, :] = se_theta[:hetero_size].reshape(n_hetero, n_classes, order="F")
        ci_beta[:n_hetero, :, :] = ci_theta[:hetero_size].reshape(n_hetero, n_classes, 2, order="F")

    if n_common > 0:
        se_common = se_theta[hetero_size:hetero_size + n_common]
        ci_common = ci_theta[hetero_size:hetero_size + n_common]
        se_beta[n_hetero:, :] = se_common[:, None]
        ci_beta[n_hetero:, :, :] = ci_common[:, None, :]

    print(output.get('message', ''))
    print()
    print(f"Log-likelihood: {output['opt_ll']:.4f}")
    if n_classes > 1:
        print(f"Class probs: {np.round(output['opt_mu_prob'], 6)}")
    if robust_se:
        print("Robust standard errors, clustered on the", individual_var, "variable.")
    else:
        print("Standard errors are not robust and assume correct model specification.")
    print()

    widths = (12, 10, 11, 10, 10)
    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'[Confidence Interval]':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)

    if n_common > 0:
        print("Common parameters")
        print(header_top)
        print(divider)
        for i, name in enumerate(covariate_vars[n_hetero:], start=n_hetero):
            row = " | ".join([
                f"{name:<{widths[0]}}",
                f"{B_hat[i, 0]:>{widths[1]}.6f}",
                f"{se_beta[i, 0]:>{widths[2]}.6f}",
                f"{ci_beta[i, 0, 0]:>{widths[3]}.5f}",
                f"{ci_beta[i, 0, 1]:>{widths[4]}.5f}"
            ])
            print(row)
        print()

    for c in range(n_classes):
        print(f"Class {c + 1}")
        print(f"Estimated proportion: {output['opt_mu_prob'][c]:.6f}")
        print(header_top)
        print(divider)

        for i, name in enumerate(covariate_vars[:n_hetero]):
            row = " | ".join([
                f"{name:<{widths[0]}}",
                f"{B_hat[i, c]:>{widths[1]}.6f}",
                f"{se_beta[i, c]:>{widths[2]}.6f}",
                f"{ci_beta[i, c, 0]:>{widths[3]}.5f}",
                f"{ci_beta[i, c, 1]:>{widths[4]}.5f}"
            ])
            print(row)
        print()


def _mle_estimator(df,
                   choice_var,
                   covariate_vars,
                   individual_var,
                   n_alts,
                   n_classes,
                   heterogenous_vars,
                   beta_init,
                   mu_init,
                   ci_alpha,
                   robust_se,
                   opt_method):
    """
    Estimates the multinomial logit model with discrete heterogeneity using maximum likelihood estimation,
    from initial values (beta_init, mu_init).
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    individual_ids = df[individual_var].to_numpy()
    k = X.shape[1]

    if heterogenous_vars is None:
        n_hetero = k
    else:
        n_hetero = len(heterogenous_vars)
        if covariate_vars[:n_hetero] != list(heterogenous_vars):
            raise ValueError("heterogenous_vars must be the first entries of covariate_vars.")

    theta_init = np.concatenate([beta_init.reshape(-1, order="F"), mu_init])

    result = minimize(
        fun=_negloglik,
        x0=theta_init,
        args=(X, Y, individual_ids, n_alts, n_classes, n_hetero),
        method=opt_method,
    )

    opt_ll = -result.fun
    opt_theta = result.x
    B_hat, mu_free_hat = _unpack_theta(opt_theta, k, n_classes, n_heterogenous=n_hetero)
    mu_raw_hat = np.concatenate(([0.0], mu_free_hat)) if n_classes > 1 else np.array([0.0])
    mu_prob_hat = np.exp(mu_raw_hat - logsumexp(mu_raw_hat))

    H = approx_hess(opt_theta, _negloglik, args=(X, Y, individual_ids, n_alts, n_classes, n_hetero))
    H_inv = np.linalg.pinv(H)
    se_theta = np.sqrt(np.clip(np.diag(H_inv), 0.0, None))  # avoid negative variances from numerical issues

    z_score = norm.ppf(1 - ci_alpha / 2)
    ci_theta = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta)]

    output = {
        'opt_ll': opt_ll,
        'opt_theta': opt_theta,
        'opt_beta': B_hat,
        'opt_mu_raw': mu_raw_hat,
        'opt_mu_prob': mu_prob_hat,
        'se_theta': se_theta,
        'ci_theta': ci_theta,
        'success': result.success,
        'message': result.message,
    }

    if robust_se:
        # J estimated by outer products of household-level scores.
        S = score_matrix(opt_theta, X, Y, individual_ids, n_alts, n_classes, n_heterogenous=n_hetero)
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_theta_r = np.sqrt(np.clip(np.diag(V_robust), 0.0, None))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta_r)]

        output['se_theta'] = se_theta_r
        output['ci_theta'] = ci_r

    return output


def estimate_MNL(df,
                 choice_var,
                 covariate_vars,
                 individual_var,
                 n_alts,
                 n_classes=1,
                 heterogenous_vars=None,
                 beta_init=None,
                 mu_init=None,
                 ci_alpha=0.05,
                 robust_se=False,
                 randomized_starts=0,
                 search_bounds=[-10, 10],
                 seed=None,
                 opt_method='BFGS'):
    """
    Estimates the multinomial logit model with discrete heterogeneity using maximum likelihood estimation.
    
    Optional randomized starts are drawn uniformly from the specified `search_bounds` around (`beta_init`, `mu_init`) 
    to help avoid local optima.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        A list of column names for the covariates.
    individual_var : str
        The column name of the individual ID variable.
        If robust standard errors are requested, standard errors are clustered by this variable.
    n_alts : int
        Number of alternatives.
    n_classes : int
        Number of latent classes. 
        If 1 (default), reduces to standard MNL.
    heterogenous_vars : list or None
        A list of column names for covariates with class-varying coefficients.
        If None (default), all covariates are assumed to have class-varying coefficients
    beta_init : array, optional
        Initial guess for coefficients. 
        If None, defaults to zeros.
    mu_init : array, optional
        Initial guess for class probabilities. 
        If None, defaults to zeros.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors. 
        If True, standard errors are clustered by `individual_var`.
    randomized_starts : int
        Number of additional random starting points to use for optimization.
    search_bounds : list
        Bounds for random perturbations around initial values, specified as [lower_bound, upper_bound].
    seed : int or None
        Random seed for reproducibility of randomized starts.
    opt_method : str
        Optimization method.

    Returns    
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - coefficient estimates
        - standard errors
        - confidence intervals for coefficients
    """

    # Determine number of covariate parameters to estimate
    if heterogenous_vars is not None:
        k = len(heterogenous_vars) * n_classes + (len(covariate_vars) - len(heterogenous_vars))
    else:
        k = len(covariate_vars) * n_classes

    # Base start: user-provided if available, otherwise zeros.
    if beta_init is None:
        beta_base = np.zeros(k)
    else:
        beta_base = np.asarray(beta_init).reshape(-1)

    if n_classes == 1:
        mu_base = np.array([])
    else:
        if mu_init is None:
            mu_base = np.zeros(n_classes - 1)
        else:
            mu_base = np.asarray(mu_init).reshape(-1)

    # Generate randomized starts
    rng = np.random.default_rng(seed)
    n_rand = max(int(randomized_starts), 0)
    beta_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, k))
    beta_rand = beta_base[None, :] + beta_perturbs  # (n_rand, k)

    if n_classes > 1:
        mu_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, n_classes - 1))
        mu_rand = mu_base[None, :] + mu_perturbs  # (n_rand, n_classes - 1)
    else:
        mu_rand = np.empty((n_rand, 0))  # (n_rand, 0)
    
    starts = [(beta_base.copy(), mu_base.copy())] + list(zip(beta_rand, mu_rand))

    best_output = None
    for beta0, mu0 in starts:
        out = _mle_estimator(df=df,
                             choice_var=choice_var,
                             covariate_vars=covariate_vars,
                             individual_var=individual_var,
                             n_alts=n_alts,
                             n_classes=n_classes,
                             heterogenous_vars=heterogenous_vars,
                             beta_init=beta0,
                             mu_init=mu0,
                             ci_alpha=ci_alpha,
                             robust_se=robust_se,
                             opt_method=opt_method)
        if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
            best_output = out

    print_mle_output(best_output, covariate_vars, 
                     heterogenous_vars=heterogenous_vars, 
                     robust_se=robust_se, individual_var=individual_var)
    
    return best_output

In [5]:
# 2 latent classes, heterogeneity on brand intercepts

results = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=2,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4'],
    robust_se=True,
    randomized_starts=3,
    opt_method='BFGS',
)

Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7923.7641
Class probs: [0.165823 0.834177]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
f            |   0.286214 |    0.135617 |    0.02041 |    0.55202
p            | -33.123372 |    3.874691 |  -40.71763 |  -25.52912
prev_purch   |  -0.202985 |    0.070645 |   -0.34145 |   -0.06452

Class 1
Estimated proportion: 0.165823
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   1.864504 |    0.822348 |    0.25273 |    3.47628
brand_2      |   2.349280 |    1.072809 |    0.24661 |    4.45195
brand_3      |  -2.124409 |    1.045900 |   -4.17434 |   -0.07448
brand_4      |   2.608096 |    0.448559 |    1.72894 |    3.48726

Class 2
Estimated proportion: 0.83417

In [6]:
# 2 latent classes, heterogeneity on all covariates

results = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=2,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    robust_se=True,
    randomized_starts=3,
    opt_method='BFGS',
)

Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7870.1579
Class probs: [0.668296 0.331704]
Robust standard errors, clustered on the hh variable.

Class 1
Estimated proportion: 0.668296
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   1.782859 |    0.359386 |    1.07848 |    2.48724
brand_2      |   2.118326 |    0.303083 |    1.52430 |    2.71236
brand_3      |  -2.011919 |    0.307475 |   -2.61456 |   -1.40928
brand_4      |  -0.387892 |    0.406693 |   -1.18500 |    0.40921
f            |   0.098548 |    0.197326 |   -0.28820 |    0.48530
p            | -45.477316 |    3.921576 |  -53.16346 |  -37.79117
prev_purch   |  -0.219414 |    0.103313 |   -0.42190 |   -0.01692

Class 2
Estimated proportion: 0.331704
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |  -

In [7]:
# 4 latent classes, heterogeneity on brand intercepts

results = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=4,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4'],
    robust_se=True,
    randomized_starts=3,
    opt_method='BFGS',
)

Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7090.9437
Class probs: [0.415197 0.099487 0.047378 0.437938]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
f            |   0.370323 |    0.132180 |    0.11126 |    0.62939
p            | -37.094802 |    3.581721 |  -44.11485 |  -30.07476
prev_purch   |  -0.109008 |    0.067040 |   -0.24040 |    0.02239

Class 1
Estimated proportion: 0.415197
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |  -0.015109 |    0.423237 |   -0.84464 |    0.81442
brand_2      |   0.977523 |    0.324199 |    0.34210 |    1.61294
brand_3      |  -3.115800 |    0.361304 |   -3.82394 |   -2.40766
brand_4      |  -1.349592 |    0.516152 |   -2.36123 |   -0.33795

Class 2
Estimated p

## Simulations

In this section, I simulate synthetic data to verify that the `estimate_mle` function can correctly uncover the underlying parameters. I also test the behavior of the function when an unnecessary latent class is estimated (i.e., the model is not point-identified).

In [8]:
# Simulate synthetic finite-mixture MNL data and check parameter recovery
from itertools import permutations

np.random.seed(94305)

# Problem dimensions
n_classes = 3
n_alts = 4
n_covariates = 5
n_hetero = 3
n_common = n_covariates - n_hetero
n_households = 500
n_choices_per_hh = 20
n_choices = n_households * n_choices_per_hh

# True parameters
B_true = np.zeros((n_covariates, n_classes))
B_true[:n_hetero, 0] = [2.0, -0.5, 0.5]
B_true[:n_hetero, 1] = [-1.0, -1.5, 1.5]
B_true[:n_hetero, 2] = [0.5, 1.0, -1.0]
B_true[n_hetero:, :] = [[-2.0], [0.8]]  # common coefficients
mu_true = np.array([0.2, 0.5, 0.3])

# Assign households to latent classes
hh_ids = np.arange(n_households)
n_c = (mu_true * n_households).astype(int)
class_by_hh = np.empty(n_households, dtype=int)
class_by_hh[:n_c[0]] = 0
class_by_hh[n_c[0]:n_c[0] + n_c[1]] = 1
class_by_hh[n_c[0] + n_c[1]:] = 2

# Simulate each choice-occasion
hh_by_choice = np.repeat(hh_ids, n_choices_per_hh)
choice_id = np.arange(n_choices)
class_by_choice = class_by_hh[hh_by_choice]
X_all = np.random.normal(size=(n_choices, n_alts, n_covariates))
B_by_choice = B_true[:, class_by_choice].T  # (NT, K)
V_inside = np.einsum("ntk,nk->nt", X_all, B_by_choice)  # (NT, J)
V0 = np.concatenate([np.zeros((n_choices, 1)), V_inside], axis=1)
probs = np.exp(V0 - logsumexp(V0, axis=1, keepdims=True))
u = np.random.rand(n_choices, 1)
cumprobs = np.cumsum(probs, axis=1)
choices = (u > cumprobs).sum(axis=1)

# Build dataset (long format)
y_inside = (choices[:, None] == np.arange(1, n_alts + 1)).astype(int)
alt = np.tile(np.arange(1, n_alts + 1), n_choices)
x_flat = X_all.reshape(n_choices * n_alts, n_covariates)
df_sim = pd.DataFrame({
    "hh": np.repeat(hh_by_choice, n_alts),
    "choice_id": np.repeat(choice_id, n_alts),
    "alt": alt,
    "y": y_inside.reshape(-1),
    "x1": x_flat[:, 0],
    "x2": x_flat[:, 1],
    "x3": x_flat[:, 2],
    "x4": x_flat[:, 3],
    "x5": x_flat[:, 4],
})

In [9]:
# Print true parameters for reference
print("True B parameters:")
print("Common coefficients:", B_true[n_hetero:, 0])
for c in range(n_classes):
    print(f"Class {c + 1}: {B_true[:n_hetero, c]}")
print(f"True class probabilities: {mu_true}")

# Estimate
sim_results = estimate_MNL(
    df=df_sim,
    choice_var="y",
    covariate_vars=["x1", "x2", "x3", "x4", "x5"],
    individual_var="hh",
    n_alts=n_alts,
    n_classes=n_classes,
    heterogenous_vars=["x1", "x2", "x3"],
    robust_se=True,
    randomized_starts=3,
    search_bounds=[-5.0, 5.0],
    seed=94305,
    opt_method="BFGS",
)

B_hat = sim_results["opt_beta"]
mu_hat = sim_results["opt_mu_prob"]

True B parameters:
Common coefficients: [-2.   0.8]
Class 1: [ 2.  -0.5  0.5]
Class 2: [-1.  -1.5  1.5]
Class 3: [ 0.5  1.  -1. ]
True class probabilities: [0.2 0.5 0.3]
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7658.0457
Class probs: [0.200011 0.499999 0.29999 ]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x4           |  -2.007901 |    0.025841 |   -2.05855 |   -1.95725
x5           |   0.818937 |    0.019327 |    0.78106 |    0.85682

Class 1
Estimated proportion: 0.200011
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x1           |   2.001888 |    0.048357 |    1.90711 |    2.09666
x2           |  -0.525658 |    0.043622 |   -0.61116 |   -0.44016
x3           |   0.535457 |    0.037979 |    0.46102 |   

In [10]:
# Behavior when an unnecessary class is added

# Print true parameters for reference
print("True B parameters:")
print("Common coefficients:", B_true[n_hetero:, 0])
for c in range(n_classes):
    print(f"Class {c + 1}: {B_true[:n_hetero, c]}")
print(f"True class probabilities: {mu_true}")

# Estimate
sim_results = estimate_MNL(
    df=df_sim,
    choice_var="y",
    covariate_vars=["x1", "x2", "x3", "x4", "x5"],
    individual_var="hh",
    n_alts=n_alts,
    n_classes=n_classes + 1,
    heterogenous_vars=["x1", "x2", "x3"],
    robust_se=True,
    randomized_starts=3,
    search_bounds=[-5.0, 5.0],
    seed=94305,
    opt_method="BFGS",
)

True B parameters:
Common coefficients: [-2.   0.8]
Class 1: [ 2.  -0.5  0.5]
Class 2: [-1.  -1.5  1.5]
Class 3: [ 0.5  1.  -1. ]
True class probabilities: [0.2 0.5 0.3]
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7658.0457
Class probs: [0.29999  0.499999 0.200011 0.      ]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x4           |  -2.007901 |    0.025841 |   -2.05855 |   -1.95725
x5           |   0.818937 |    0.019327 |    0.78106 |    0.85682

Class 1
Estimated proportion: 0.299990
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x1           |   0.502738 |    0.029812 |    0.44431 |    0.56117
x2           |   0.993590 |    0.032877 |    0.92915 |    1.05803
x3           |  -1.010410 |    0.032465 |   -1.0